# レッスン18: 暗号化レシートによるAIエージェントのセキュリティ強化

## ハンズオンノートブック

このノートブックでは4つの演習を通じて進めます：

1. エージェントツール呼び出しのために<strong>最初のレシートに署名し、検証する</strong>。
2. **レシートを改ざんし、検証が失敗する様子を見る**。
3. **3つのレシートのチェーンを構築し、チェーンの整合性を確認する**。
4. **Microsoft Agent Frameworkツール呼び出しをラップし、すべての操作がレシートを発行するようにする**。

すべての暗号プリミティブは、よく管理されたライブラリ（Ed25519用の`pynacl`、RFC 8785標準の正規JSON用の`jcs`、SHA-256用のPython標準ライブラリの`hashlib`）からインポートしています。レシートのロジック自体は、読み書き・改変可能な純粋なPythonです。

セルは順に実行してください。各セクションは短く独立しています。


## Setup

2つの依存関係をインストールします。両方とも寛容なライセンス（Apache-2.0 / MIT）です。


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ヘルパーユーティリティ

これら2つのヘルパーは、base64urlエンコーディング（パディングなし）と任意のオブジェクトのSHA-256ハッシュ化を処理します。これにより、ノートブックの他の部分はレシートロジック自体に集中できます。


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: 最初のレシートに署名する

**Contoso Travel** のエージェントが顧客のためにシドニーからロサンゼルスへの航空便を調べたと想像してください。このツール呼び出しを署名済みレシートとして記録し、将来の監査人が私たちを信頼せずに検証できるようにしたいと思います。

### Step 1.1: 署名キーを生成する

本番環境では、エージェントの署名キーはハードウェアセキュリティモジュール（HSM）、Azure Key Vault、または同様の保護されたストアに保存されます。このレッスンでは、新しいキーをメモリ内で生成します。


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ステップ 1.2: レシートペイロードの作成

ペイロードには、レシートが証明するすべての情報が含まれます。つまり、誰が行動し、どのツールを使い、どの引数で、何が返され、どのポリシーの下で、いつ行われたかです。引数と結果はインラインで含めるのではなくハッシュ化しているため、レシートが機密情報を漏らすことはありません。


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: 受領書に署名して組み立てる

3つのステップ：

1. 2つの実装が同じ論理的な受領書を生成した場合、バイトも同一になるようにJCSを使ってペイロードを正規化する。
2. 正規化されたバイト列をSHA-256でハッシュ化する。
3. ハッシュにEd25519秘密鍵で署名する。

その後、署名は元のペイロードに添付されて最終的な受領書が作成される。


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ステップ 1.4: 受領書の検証

検証はこのプロセスを逆に行います。署名を取り除き、正規化されたハッシュを再計算し、受領書内の公開鍵に対して署名を検証します。

この検証を行う監査人は、私たちから受領書自体以外は何も必要としません。呼び出すサービスも、照会する鍵ディレクトリも、信頼も必要ありません。


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

`Receipt is valid: True` と表示されているはずです。エージェントは最初の暗号的に署名された監査記録を作成しました。


## セクション 2: レシートの改ざん

レシートの目的は改ざんが分かることにあります。これを証明しましょう。

レシートの文字を1つだけ変更し、検証が失敗することを確認します。


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 何が起こったのか？

`policy_id` を変更したとき、正規化されたバイト列が変わりました。そのバイト列の SHA-256 ハッシュも変わりました。署名（元のハッシュに対して作成されたもの）は新しいハッシュと一致しなくなりました。検証は正しく `False` を返します。

攻撃者が秘密鍵を持っていない限り、領収書のどのフィールドを変更しても検証は通りません。秘密鍵がキーコンテナーにあり公開鍵が公開されている限り、改ざんを隠すことは不可能です。

実際に試してみてください: 上のセルの `tool_name` や `agent_id`、`timestamp` を変更して再実行してみてください。どの変更も無効な領収書を生み出します。


## セクション 3: レシートを連結する

1 つのレシートは 1 つのアクションを保護します。ほとんどのエージェントは多くのアクションを実行します。シーケンス全体を改ざん検出可能にするために、各レシートに前のレシートのハッシュを新しいレシートのペイロードに含めて、前のレシートにリンクさせます。

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

誰かがレシートを削除したり並べ替えたりすると、ちょうどそのポイントでチェーンが切れます。その後の任意のレシートの検証は、その `previous_receipt_hash` が前のレシートの実際のハッシュと一致しなくなるため失敗します。


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

次に、中間のレシートを改ざんしてチェーンを切り、再度検証します。改ざんされたレシートは署名チェックに失敗し、さらに次のレシートはチェーンリンクチェックに失敗します（`previous_receipt_hash` が変更された中間レシートのハッシュと一致しなくなるため）。


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

レシート0はまだ検証に成功します（変更されておらず、依存する前のレシートもありません）。レシート1は `tool_args_hash` を変更したため、署名の検証に失敗します。レシート2は `previous_receipt_hash` が元の（現在は変更された）レシート1に対して計算されているため、チェーンリンクの検証に失敗します。

攻撃者が変更されたレシート1に再署名したとしても（秘密鍵がなければできません）、レシート2のチェーンリンクの不一致によって改ざんが露見します。変更を隠すためには、攻撃者は変更点以降のすべてのレシートに再署名する必要があり、それには秘密鍵の所持が必要です。


## セクション 4: 代理ツールの呼び出しを領収書署名でラップする

実際の運用では、すべての代理の作者が `make_receipt` を呼び出すことを覚えておく必要はありません。すべてのツール呼び出しに対して領収書署名が自動的に行われることを望みます。

ここに最も単純なパターンがあります：任意の呼び出し可能なツール関数を受け取り、それの領収書を発行するバージョンを返すラッパークラスです。これは Microsoft Agent Framework (`agent_framework.azure`) を含む任意の代理フレームワークに適応します。

Azure AI Foundry プロジェクトを設定していなくても、以下のローカルモックはこのパターンを示しています。


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Frameworkとの統合

上記の`ReceiptedTool`ラッパーはフレームワークに依存しません。Microsoft Agent Frameworkで構築されたエージェント内で使用するには、ラップされた関数をツールとして登録します。以下はスケッチ（モックを実際のAzure AI Foundryツール登録に置き換えます）です:

```python
# 統合の形状を示す擬似コード。
# agent_framework.azure から AzureAIProjectAgentProvider をインポート
# azure.identity から AzureCliCredential をインポート
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="あなたは Contoso Travel エージェントです ...",
#     tools=[receipted_lookup],   # 生の関数ではなくラップされたツール
# )
# response = agent.run("6月にシドニーからロサンゼルスへのフライトを探してください。")
#
# # 実行後、エージェントが行ったすべてのツール呼び出しには署名付きのレシートがあります：
# audit_chain = receipted_lookup.receipts
```

エージェントフレームワークはレシートについて何も知る必要はありません。レシート署名はツールの周りにラップされており、フレームワークに組み込まれているわけではありません。これにより、エージェントを再構築することなく既存のエージェントコードに出所情報を追加できます。


## 振り返りと追加チャレンジ

あなたは以下を行いました：

- Ed25519の鍵ペアを生成した。
- エージェントツール呼び出しのレシートを作成して署名した。
- 公開鍵だけを用いてレシートをオフラインで検証した。
- レシートを改ざんして検証失敗を確認した。
- 3つのレシートのハッシュチェーンを構築した。
- チェーンの途中を改ざんし、署名の失敗とチェーンリンクの失敗の両方を確認した。
- ツール関数を自動レシート署名でラップした。

**追加チャレンジ。** レシートスキーマに `request_id` フィールド（分散トレーシング用のUUID）を追加してください。`make_receipt` を更新してそれを含め、レシートが引き続き端から端まで検証可能であることを確認してください。その後、署名後にこのフィールドを変更し、検証に失敗することを確認してください。これは、正準エンコーディングのすべてのバイトが署名にどのように関与しているかを理解させるためのものです。

**重要な境界。** レシートが証明するのは3つの事実、かつその3つだけです：帰属（この鍵がこの内容に署名した）、完全性（署名以来内容が変わっていない）、順序（このレシートはあのレシートの後に来た）。レシートは、エージェントの行動が正しいこと、`policy_id` に名前があるポリシーが実際に評価されたこと、エージェントがすべてのルールに従ったことを証明しません。レシートは基盤です。ガバナンスはその上に構築するシステムです。

この境界を念頭に置いてレッスンのREADMEを再度読みましょう。チームがレシートに関して犯す最も一般的な誤りは、「レシートがある」ということを「私たちはガバナンスされている」と誤解することです。そうではありません。レシートはエージェントの振る舞いを監査可能にしますが、正しいことを保証するものではありません。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
